[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/Zgraph/blob/main/zgraph/examples/Extruded_dimension.ipynb)

In [ ]:
# Install Zgraph if running in Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/themintlab/Zgraph.git
    print("Successfully installed Zgraph!")

In [ ]:
import torch
from zgraph.core import FactorNode, SignalNode, SignalNodes, ProductNode, ConstantNode, LeafNode
from zgraph.transforms import finalize, legendre_transform
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
T, mu1, mu2, p = SignalNodes(0, 1, 2, 3)

In [ ]:
R = 8.314
RT = FactorNode([[R]], [T])
mu1A = FactorNode([2, -1], [RT, mu1] )
mu2A = FactorNode([-1], [mu2])
mu1B = FactorNode([-1], [mu1])
mu2B = FactorNode([1, -1], [RT, mu2] )

In [ ]:
phaseA = FactorNode(torch.eye(2), [mu1A, mu2A], beta=RT)
phaseB = FactorNode(torch.eye(2), [mu1B, mu2B], beta=RT)
system = FactorNode(, [phaseA, phaseB], beta=0)
system = FactorNode(torch.eye(2), [phaseA, phaseB], beta=0)

In [ ]:
phaseAb, phaseBb, systemb = finalize([phaseA, phaseB, system], compile_graph=False)

fcns = legendre_transform([phaseA, phaseB, system], [1, 2])
fA, fB, fsys = finalize(fcns, compile_graph=False)

fAd, cAd = fA(input_tensor)
fBd, cBd = fB(input_tensor)
fsysd, csysd = fsys(input_tensor)

# Grand potential plot

In [ ]:
T_val = torch.tensor(298.15)
mu1 = torch.linspace(-10 * R * 300, 10 * R * 300, steps=500)
mu2 = -mu1
T_flat = T_val.expand_as(mu1)
P_flat = torch.ones_like(mu1)
input_tensor = torch.stack([T_flat, mu1, mu2, P_flat], dim=-1)

In [ ]:
gA_vals = phaseAb(input_tensor).detach().cpu().numpy().squeeze()
gB_vals = phaseBb(input_tensor).detach().cpu().numpy().squeeze()
gsys_vals = systemb(input_tensor).detach().cpu().numpy().squeeze()
x_mu = input_tensor[..., 1].detach().cpu().numpy().squeeze()

# Free energy plot

In [ ]:
xA_frac = -cAd[:, 1].detach().cpu().numpy().squeeze()
yA_free = -fAd.detach().cpu().numpy().squeeze()

xB_frac = -cBd[:, 1].detach().cpu().numpy().squeeze()
yB_free = -fBd.detach().cpu().numpy().squeeze()

xsys_frac = -csysd[:, 1].detach().cpu().numpy().squeeze()
ysys_free = -fsysd.detach().cpu().numpy().squeeze()

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.12,
    subplot_titles=("Grand potential plot", "Free energy plot")
)

colors = {
    "phase A": "#1f77b4",
    "phase B": "#ff7f0e",
    "Equilibrium": "#2ca02c"
}

fig.add_trace(
    go.Scatter(
        x=x_mu, y=gA_vals,
        name="phase A",
        legendgroup="phase A",
        showlegend=True,
        line=dict(width=3, color=colors["phase A"])
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=x_mu, y=gB_vals,
        name="phase B",
        legendgroup="phase B",
        showlegend=True,
        line=dict(width=3, color=colors["phase B"])
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=x_mu, y=gsys_vals,
        name="Equilibrium",
        legendgroup="Equilibrium",
        showlegend=True,
        line=dict(width=3, color=colors["Equilibrium"])
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=xA_frac, y=yA_free,
        name="phase A",
        legendgroup="phase A",
        showlegend=False,
        mode="lines+markers",
        marker=dict(size=4),
        line=dict(width=3, color=colors["phase A"])
    ),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=xB_frac, y=yB_free,
        name="phase B",
        legendgroup="phase B",
        showlegend=False,
        mode="lines+markers",
        marker=dict(size=4),
        line=dict(width=3, color=colors["phase B"])
    ),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=xsys_frac, y=ysys_free,
        name="Equilibrium",
        legendgroup="Equilibrium",
        showlegend=False,
        mode="lines+markers",
        marker=dict(size=4),
        line=dict(width=3, color=colors["Equilibrium"])
    ),
    row=2, col=1
)

fig.update_xaxes(title_text="Chemical potential difference, Δμ", row=1, col=1)
fig.update_yaxes(title_text="Grand potential, Ω", row=1, col=1)

fig.update_xaxes(title_text="Mole fraction", row=2, col=1)
fig.update_yaxes(title_text="Free energy", row=2, col=1)

fig.update_layout(
    height=750,
    width=800,
    template="plotly_white"
)

fig.show()